In [ ]:
!pip -q install torch torchvision

import os
import random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision.transforms import functional as TF

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/data.zip"
extract_path = "/content/data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped to:", extract_path)

Unzipped to: /content/data


In [ ]:
DATASET_DIR = "/content/data/data"

In [ ]:
import os

print("DATASET_DIR:", DATASET_DIR)
files = os.listdir(DATASET_DIR)
print("First 10 files:", files[:10])

img_files = [f for f in files if f.endswith("_img.png")]
print("Number of _img files:", len(img_files))

DATASET_DIR: /content/data/data
First 10 files: ['0508_mask.png', '0994_mask.png', '0567_img.png', '0898_img.png', '0833_mask.png', '0869_img.png', '0150_mask.png', '0419_img.png', '0566_mask.png', '0589_mask.png']
Number of _img files: 1000


In [ ]:
class SegDataset(Dataset):
    def __init__(self, root_dir, train=True):
        self.root_dir = root_dir
        self.train = train

        self.image_paths = sorted([
            os.path.join(root_dir, f)
            for f in os.listdir(root_dir)
            if f.endswith("_img.png")
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = img_path.replace("_img.png", "_mask.png")

        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # Resize for training speed (model will upscale back)
        img = TF.resize(img, (320, 448))
        mask = TF.resize(mask, (320, 448))

        if self.train:
            if random.random() < 0.5:
                img = TF.hflip(img)
                mask = TF.hflip(mask)

        img = TF.to_tensor(img)   # [0,1]
        mask = TF.to_tensor(mask)
        mask = (mask > 0.5).float()

        return img, mask

In [ ]:
dataset = SegDataset(DATASET_DIR, train=True)

val_size = int(0.1 * len(dataset))
train_size = len(dataset) - val_size

train_ds, val_ds = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

print("Total:", len(dataset))
print("Train:", len(train_ds))
print("Val:", len(val_ds))

Total: 1000
Train: 900
Val: 100


In [ ]:
img, mask = dataset[0]

print("Image shape:", img.shape)
print("Mask shape:", mask.shape)
print("Image min/max:", img.min().item(), img.max().item())
print("Mask min/max:", mask.min().item(), mask.max().item())

Image shape: torch.Size([3, 320, 448])
Mask shape: torch.Size([1, 320, 448])
Image min/max: 0.0 0.9803921580314636
Mask min/max: 0.0 1.0


In [ ]:
class FastSegModel(nn.Module):
    def __init__(self):
        super().__init__()

        weights = torchvision.models.segmentation.LRASPP_MobileNet_V3_Large_Weights.DEFAULT
        base = torchvision.models.segmentation.lraspp_mobilenet_v3_large(weights=weights)

        # Replace final classifier heads to output 1 channel
        # LRASPPHead has low_classifier and high_classifier
        low_in = base.classifier.low_classifier.in_channels
        high_in = base.classifier.high_classifier.in_channels

        base.classifier.low_classifier = nn.Conv2d(low_in, 1, kernel_size=1)
        base.classifier.high_classifier = nn.Conv2d(high_in, 1, kernel_size=1)

        self.model = base

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        h, w = x.shape[-2:]

        x = (x - self.mean) / self.std
        out = self.model(x)["out"]
        out = F.interpolate(out, size=(h, w), mode="bilinear", align_corners=False)
        out = torch.sigmoid(out)

        return out

model = FastSegModel().to(device)
print("Fast model created successfully")

Downloading: "https://download.pytorch.org/models/lraspp_mobilenet_v3_large-d234d4ea.pth" to /root/.cache/torch/hub/checkpoints/lraspp_mobilenet_v3_large-d234d4ea.pth


100%|██████████| 12.5M/12.5M [00:00<00:00, 149MB/s]

Fast model created successfully


In [ ]:
def loss_fn(pred, target):
    bce = F.binary_cross_entropy(pred, target)

    inter = (pred * target).sum()
    union = pred.sum() + target.sum()

    dice = (2 * inter + 1e-6) / (union + 1e-6)

    return bce + (1 - dice)

def compute_iou(pred, target):
    inter = torch.minimum(pred, target).sum()
    union = torch.maximum(pred, target).sum()
    return (inter / (union + 1e-6)).item()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
OUTPUT_DIR = "/content/output"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
EPOCHS = 10
best_iou = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        preds = model(imgs)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_iou = 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            val_iou += compute_iou(preds, masks)

    val_iou /= len(val_loader)

    print(f"Epoch {epoch+1}: Loss {train_loss:.3f}, Val IoU {val_iou:.4f}")

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), OUTPUT_DIR + "/best_fast.pth")

Epoch 1: Loss 21.758, Val IoU 0.9353
Epoch 2: Loss 13.669, Val IoU 0.9509
Epoch 3: Loss 11.158, Val IoU 0.9586
Epoch 4: Loss 9.794, Val IoU 0.9628
Epoch 5: Loss 8.894, Val IoU 0.9663
Epoch 6: Loss 8.260, Val IoU 0.9682
Epoch 7: Loss 7.798, Val IoU 0.9704
Epoch 8: Loss 7.447, Val IoU 0.9713
Epoch 9: Loss 7.095, Val IoU 0.9726
Epoch 10: Loss 6.873, Val IoU 0.9736


In [ ]:
model.load_state_dict(torch.load(OUTPUT_DIR + "/best_fast.pth", map_location=device))
model.eval()
print("Best fast model loaded")

Best fast model loaded


In [ ]:
dummy = torch.randn(1, 3, 436, 600).to(device)

traced_model = torch.jit.trace(model, dummy)
traced_model.save(OUTPUT_DIR + "/model.pt")

print("Saved fast TorchScript model")

Saved fast TorchScript model


In [ ]:
m = torch.jit.load(OUTPUT_DIR + "/model.pt", map_location=device)
m.eval()

x = torch.rand(1, 3, 436, 600).to(device)
y = m(x)

print("Shape:", y.shape)
print("Min:", y.min().item())
print("Max:", y.max().item())

Shape: torch.Size([1, 1, 436, 600])
Min: 0.00021148087398614734
Max: 0.02236650139093399


In [ ]:
from google.colab import files
files.download(OUTPUT_DIR + "/model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>